# An Introduction to Git

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*

An *interactive* guide: every command below runs in a small sandbox repository
created by the notebook itself.

## What you will learn

- What version control is and why you need it
- The three places a file can be: working directory, staging area, repository
- The everyday cycle: `status` &rarr; `add` &rarr; `commit`
- Looking at history and at differences
- Undoing things
- Branches, merges, and merge conflicts
- Working with GitHub: `clone`, `push`, `pull`, forks and pull requests

## 1. Why version control?

Without it, a project ends up looking like this:

```
thesis_v2.tex   thesis_v2_final.tex   thesis_v2_final_REAL.tex
analysis.py     analysis_old.py       analysis_works.py
```

Git gives you instead:

- A **complete history**: every version, who changed what, and why.
- The ability to **go back** to any earlier state.
- A way for several people to work on the same files **without overwriting each other**.
- A **backup**, once the project is on a server such as GitHub.

💡 Think of Git as a *time machine* for your project.

## 2. The model: three places

A file in a Git project lives in one of three places.

```
   working directory  --- git add --->  staging area  --- git commit --->  repository
   (what you edit)                      (what will go                     (the permanent
                                         in the next commit)               history)
```

- The **working directory** is the folder you actually edit.
- The **staging area** (or *index*) is where you assemble the next commit.
- The **repository** is the `.git` folder: the full history, stored locally.

A **commit** is a snapshot of the whole project, with a message, an author and a date.

## 3. A sandbox to play in

Everything happens inside `git_sandbox`, so nothing else on your computer is touched.
Running this cell again starts from a clean state.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())
os.chdir(BASE)
shutil.rmtree("git_sandbox", ignore_errors=True)
os.makedirs("git_sandbox")
os.chdir("git_sandbox")
print("Working inside:", os.path.basename(os.getcwd()) + "/")

## 4. Who are you?

Every commit records an author. You set this **once per computer**:

```bash
git config --global user.name  "Your Name"
git config --global user.email "your@email.com"
```

Use the same email as your GitHub account, so that your commits are linked to it.

Without `--global`, the setting applies **only to the current repository** &mdash;
which is what we do below, so that this notebook does not touch your real configuration.

## 5. Creating a repository

`git init` turns an ordinary folder into a repository. It creates a hidden `.git`
folder; everything else stays exactly as it was.

The `-b main` option names the first branch `main`. Without it, older versions of
Git call it `master`, which is a common source of confusion.

In [ ]:
!git init -b main
!git config user.name  "FCA Student"
!git config user.email "student@usc.es"
!ls -a

## 6. The everyday cycle

1. `git status` &mdash; what has changed?
2. `git add file` &mdash; put it in the staging area
3. `git commit -m "message"` &mdash; record it in the history

Do this often. Small, frequent commits are far more useful than large, rare ones.

In [ ]:
!echo 'print("Hello Git")' > hello.py
!git status

Git calls `hello.py` **untracked**: it has never been committed.

In [ ]:
!git add hello.py
!git status --short

`A` means *added to the staging area*. Now we record it.

In [ ]:
!git commit -m "Add hello.py, a first script" 

### Writing good commit messages

- One short line (about 50 characters), in the imperative: *"Add"*, *"Fix"*, *"Remove"*.
- Say **why**, not what: the diff already says what changed.
- One commit = one idea. Do not mix a bug fix with a reformatting.

Bad: `changes`, `stuff`, `asdf`, `final version 3`.

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Four messages</b></p>

Which of these commit messages are useful six months later? Rewrite the bad ones.

```
fix
Fix sign of the magnetic field in the track fit
changes in analysis.py and plots and README
Add energy calibration for run 42 (new pedestal values)
```

</div>

<details>
<summary><b>Solution</b></summary>

The second and the fourth: imperative, one idea, and they say what and why. `fix` says
nothing (fix what?). The third mixes three things in one commit and only lists files,
which the diff already shows: split it into three commits.

</details>

## 7. Looking at the history

```bash
git log                      # full history
git log --oneline            # one line per commit
git log --oneline --graph    # with the branch structure
git show HEAD                # what the last commit changed
```

`HEAD` is a pointer to where you are now, normally the last commit of the current branch.

In [ ]:
!git log --oneline

## 8. Seeing what changed: `git diff`

| Command | Compares |
|---|---|
| `git diff` | working directory &harr; staging area (*not yet added*) |
| `git diff --staged` | staging area &harr; last commit (*added, not yet committed*) |
| `git diff HEAD~1 HEAD` | between two commits |

Read a diff as: `-` lines removed, `+` lines added.

In [ ]:
!echo 'print("Goodbye Git")' >> hello.py
!git diff

In [ ]:
!git add hello.py
!git commit -m "Add a farewell message" 

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Add, then edit again</b></p>

We change `hello.py`, `git add` it, and change it **again**. **Decide** what
`git status --short` shows, and what a `git commit` would record.

</div>

In [ ]:
!echo "# first change" >> hello.py
!git add hello.py
!echo "# second change" >> hello.py
!git status --short
!git diff --staged --stat && git diff --stat

<details>
<summary><b>Solution</b></summary>

`MM hello.py`: the first `M` (staging area) holds the first change; the second `M`
(working directory) the second one, not yet added. A commit records **only what is
staged**: the first change. `git add` takes a snapshot of the file at that moment, not a
reference to it.

</details>

In [ ]:
# back to a clean state (see "Undoing things" below)
!git restore --staged hello.py && git restore hello.py
!git status --short

## 9. Ignoring files: `.gitignore`

Never commit compiled output, data files, or notebook checkpoints.
List the patterns to ignore in a file called `.gitignore`:

```
__pycache__/
*.pyc
.ipynb_checkpoints/
*.root
data/
.DS_Store
```

`.gitignore` itself **is** committed, so everyone on the project ignores the same things.

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
data/

In [ ]:
!mkdir -p data && echo "10 20 30" > data/measurements.txt
!git status --short
!git add .gitignore && git commit -q -m "Add .gitignore"
!echo "(data/ is ignored, so it does not appear above)" 

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Ignored, but still there</b></p>

A file `results.dat` was committed weeks ago. Today you add `*.dat` to `.gitignore`.
Does Git stop tracking `results.dat`?

</div>

<details>
<summary><b>Solution</b></summary>

No. `.gitignore` only affects **untracked** files. A file already in the repository
keeps being tracked, and its changes keep appearing. To stop tracking it without
deleting it from your disk: `git rm --cached results.dat`, and commit.

</details>

## 10. Undoing things

| Situation | Command |
|---|---|
| Discard changes in a file (not yet added) | `git restore file` |
| Take a file out of the staging area | `git restore --staged file` |
| Change the last commit message | `git commit --amend` |
| Undo a commit, keeping the changes | `git reset --soft HEAD~1` |
| Undo a commit **and** the changes | `git reset --hard HEAD~1` ⚠️ |
| Undo a commit by making a new one | `git revert HEAD` |

> ⚠️ `reset --hard` throws work away permanently. `revert` is the safe option,
> and the only correct one if the commit has already been pushed.

In [ ]:
!echo "a mistake" >> hello.py
!git status --short
!echo "(restoring)" 
!git restore hello.py
!git status --short
!echo "(clean again)" 

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Already pushed</b></p>

You pushed a commit that broke the analysis, and two colleagues have already pulled it.
**Choose**: `git reset --hard HEAD~1` and push again, or `git revert HEAD`? Why?

</div>

<details>
<summary><b>Solution</b></summary>

`git revert HEAD`. It adds a **new** commit that undoes the bad one, so the history of
your colleagues stays valid. `reset` rewrites the history: their copies would contain a
commit that no longer exists in yours, and the next push or pull becomes a mess.
Rule: never rewrite history that others already have.

</details>

***[+] Lookout.*** `git stash` puts your uncommitted changes aside and restores them later, when you need a
clean tree for a moment; `git bisect` finds, by binary search through the history, the
commit that introduced a bug.

## 11. Branches

A branch is an independent line of work. You develop a new idea on its own branch,
and only merge it into `main` when it works.

```bash
git branch                 # list branches
git switch -c feature      # create one and move to it  (modern)
git checkout -b feature    # the same, older syntax
git switch main            # go back
```

💡 Branches are *parallel universes* of your project. `main` should always work.

In [ ]:
!git switch -c feature
!echo 'def energy(m): return m * 299792458**2' > physics.py
!git add physics.py
!git commit -q -m "Add energy function"
!git log --oneline --graph --all

## 12. Merging

`git merge other` brings the work of `other` into the branch you are on.
So: **first move to the destination**, then merge.

`--no-edit` accepts the default merge message. Without it Git opens an editor,
and inside a notebook that would hang the kernel.

There are **two possible outcomes**. Telling them apart is the whole point.


<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A merge commit?</b></p>

`main` has not changed since `feature` was created. **Decide**: will `git merge feature`
create a new commit? What will the graph look like?

</div>

In [ ]:
!git switch main
!git merge feature --no-edit
!ls
!git log --oneline --graph --all

### Case 1: fast-forward

Git printed **`Fast-forward`**, and the history is a straight line:
there is **no merge commit**.

Why? `main` had not moved since `feature` branched off, so `feature` already
contained every commit of `main`. Nothing had to be combined &mdash; Git simply slid
the `main` pointer forward.


### Case 2: the branches diverge

Now let **both** sides move. Branch `plots` adds a file, and `main` adds a
different one *after* `plots` was created. The two histories genuinely fork,
so a fast-forward is no longer possible.


In [ ]:
!git switch -c plots -q
!echo 'import matplotlib.pyplot as plt' > plotting.py
!git add plotting.py && git commit -q -m "Add plotting helper"

!git switch main -q
!echo '# project notes' > NOTES.md
!git add NOTES.md && git commit -q -m "Add project notes"

!git merge plots --no-edit
!git log --oneline --graph --all


This time Git writes a **merge commit**: a commit with *two* parents.
The graph shows the fork and the join.

```
*   Merge branch 'plots'
|\
| * Add plotting helper
* | Add project notes
|/
```

Rule of thumb: **fast-forward** when the branches never diverged, **merge commit**
when they did. Adding `--no-ff` forces a merge commit even in case 1, so the history
keeps a visible record that a branch ever existed.


## 13. Merge conflicts

A conflict happens when **the same lines** were changed on both branches.
Git cannot decide for you, so it stops and asks.

This is normal. It is not an error, and it is not a disaster.

In [ ]:
# two branches change the same line
!git switch -c experiment -q
!echo 'C = 3.0e8   # rounded' > constants.py
!git add constants.py && git commit -q -m "Add rounded constant"

!git switch main -q
!echo 'C = 299792458   # exact, m/s' > constants.py
!git add constants.py && git commit -q -m "Add exact constant"

!git merge experiment

Git leaves **markers** in the file showing both versions:

```
<<<<<<< HEAD
your version
=======
their version
>>>>>>> experiment
```

To resolve: edit the file so it says what you want, **delete the markers**,
then `git add` it and commit.

In [ ]:
!cat constants.py

In [ ]:
# resolve by hand: keep the exact value, and say so
!echo 'C = 299792458   # exact speed of light, m/s' > constants.py
!git add constants.py
!git commit -m "Merge experiment: keep the exact value of c"
!git log --oneline --graph --all

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Conflicts are a people problem</b></p>

Git found the conflict, but it could not tell which value of $c$ was right: someone had
to decide. In a team of four working on the same analysis, how would you make conflicts
rare, and easy when they happen?

</div>

<details>
<summary><b>Solution</b></summary>

Small and frequent commits; `git pull` before starting to work; short-lived branches
merged often; one person per file or module when possible; agreeing on formatting
(a formatter, so nobody "fixes" spacing in someone else's lines); and talking before
touching shared code. Conflicts grow with the time two branches live apart.

</details>

## 14. Working with a remote: GitHub

A **remote** is a copy of the repository on a server.

```bash
git clone URL                    # get a copy of an existing repository
git remote -v                    # which remotes do I have?
git remote add origin URL        # connect a local repo to a remote
git push -u origin main          # send your commits (first time)
git push                         # afterwards
git pull                         # bring in what others have pushed
```

`origin` is just the conventional name of the main remote.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Push rejected</b></p>

You commit and `git push`. Git answers `! [rejected] main -> main (fetch first)`.
**Decide** what happened and what you do. Is `git push --force` a good idea?

</div>

<details>
<summary><b>Solution</b></summary>

Someone pushed to `main` after your last pull: the remote has commits you do not have.
Do `git pull` (which fetches and merges, resolving conflicts if any), check that
everything works, and `git push` again. `--force` would overwrite their commits on the
server: never on a shared branch.

</details>

### `git clone` works with any path, not only a URL

Below we clone our sandbox repository into a second folder, to show what a clone is:
a **complete, independent copy**, history included.

In [ ]:
import os
os.chdir(BASE)
!git clone -q git_sandbox git_clone
os.chdir("git_clone")
!git log --oneline | head -3
!git remote -v

In [ ]:
import os
os.chdir(os.path.join(BASE, "git_sandbox"))
print("back in:", os.path.basename(os.getcwd()) + "/")

## 15. Authenticating with GitHub

⚠️ **GitHub stopped accepting your account password over HTTPS in 2021.**
The first time you push you will need one of these:

| Method | How |
|---|---|
| **Personal access token** | GitHub &rarr; Settings &rarr; Developer settings &rarr; Tokens. Use the token *instead of the password* when Git asks. |
| **SSH key** | `ssh-keygen -t ed25519 -C "your@email.com"`, then paste `~/.ssh/id_ed25519.pub` into GitHub &rarr; Settings &rarr; SSH keys. Clone with the `git@github.com:...` URL. |

The SSH key is more work once, and no work ever again. Set it up today, not the day you need it.

***[+] Lookout.*** The GitHub command line tool, [`gh`](https://cli.github.com), sets up the authentication
for you (`gh auth login`), and also creates repositories, forks and pull requests from
the terminal.

## 16. Collaborating: fork, branch, pull request

The workflow used by essentially every project you will contribute to:

1. **Fork** the repository on GitHub &mdash; your own copy of it.
2. **Clone** your fork to your computer.
3. Create a **branch** for what you want to do.
4. Commit, and **push** the branch to your fork.
5. Open a **pull request**: *"please take these commits into your project"*.
6. Others **review** it, you fix what they raise, and it gets **merged**.

The point of a pull request is the review. It is where most of the learning happens.

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Notebooks in Git</b></p>

Look at `git diff` of a notebook after re-running it, without changing any code. Why is it
so long? What problems does that cause in a shared repository, and what would you do?

</div>

<details>
<summary><b>Solution</b></summary>

A notebook is a JSON file that also stores the **outputs**, execution counters and
metadata: re-running it changes all of them. Diffs become unreadable, merges conflict on
outputs, the repository grows with images, and outputs can carry personal data (paths,
user names). Solutions: strip the outputs before committing
([nbstripout](https://github.com/kynan/nbstripout), which this course's repository
uses), or keep the notebook paired with a plain `.py` file
([jupytext](https://jupytext.readthedocs.io)).

</details>

## 17. Reading the history of a project

| Where | How |
|---|---|
| Terminal | `git log --oneline --graph --all` |
| GitHub | the *Commits* tab, or *Insights &rarr; Network* |
| Editor | VS Code with the *Git Graph* extension |
| Dedicated tools | GitKraken, Sourcetree |

Useful when you arrive at a project you did not write: `git log -p file.py` shows
the whole history of one file, and `git blame file.py` shows who last touched each line.

In [ ]:
!git log --oneline --graph --all

## 18. Cleaning up

Everything we made lives in `git_sandbox` and `git_clone`. This removes both.

> Run this **only when you are done**. Afterwards the sandboxes are gone, and the
> cells above will fail until you re-run the setup cell that created them.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # also works after a kernel restart
os.chdir(BASE)
for d in ("git_sandbox", "git_clone"):
    shutil.rmtree(d, ignore_errors=True)
print("Sandboxes removed. Back in:", os.path.basename(os.getcwd()) + "/")

## 19. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — Fork, branch, pull request</b></p>

In pairs.

1. Create a repository on GitHub and clone it.
2. Add a module with a function (for example, a numerical integrator) and a second
   module with a test for it. Commit and push.
3. **Fork** the repository of the pair next to you, and clone your fork.
4. On a branch, add one more function *and its test*.
5. Push the branch and open a **pull request**.
6. Review the pull request you receive: check that the tests pass and that the
   commit messages say something. Ask for a change if they do not.
7. Merge it.
8. Back on your own clone, `git pull` and check that you have their work.

Bonus: make two people change the same line on purpose, and resolve the conflict.

</div>

## 20. Summary

* Three places: working directory → `add` → staging area → `commit` → repository.
* Small commits, one idea each, with a message that says why.
* Branches for new work; merge when it works; conflicts are resolved by people.
* Never rewrite history that others already have.

<details>
<summary><b>[i] Cheat sheet — git</b></summary>

| | |
|:--|:--|
| `git init -b main` | start a repository |
| `git clone URL` | copy an existing one |
| `git status` | what has changed? |
| `git add file` &middot; `git add .` | stage changes |
| `git commit -m "..."` | record them |
| `git log --oneline --graph --all` | see the history |
| `git diff` &middot; `git diff --staged` | see the changes |
| `git restore file` | discard changes |
| `git revert HEAD` | undo a commit with a new one |
| `git switch -c name` &middot; `git switch main` | create / change branch |
| `git merge other` | bring another branch in |
| `git push` &middot; `git pull` | exchange with the remote |
| `.gitignore` | what never gets committed |

**The cycle you will use every day:** `status` &rarr; `add` &rarr; `commit` &rarr; `push`.

</details>

### Check yourself

1. Name the three places where a change can be, and the commands that move it.
2. What is the difference between `git diff` and `git diff --staged`?
3. When does a merge create a merge commit?
4. Why is `git revert` safer than `git reset --hard` for a pushed commit?
5. A file is in `.gitignore`, but Git still shows its changes. Why?

<details>
<summary><b>Solutions</b></summary>

1. Working directory → `git add` → staging area → `git commit` → repository.
2. `git diff`: changes not yet added; `git diff --staged`: added, not yet committed.
3. When both branches have new commits (they diverged); otherwise it is a fast-forward.
4. It adds a new commit instead of rewriting the history others already have.
5. It was already tracked before being ignored: `git rm --cached file`.

</details>